# Ejercicio 7: Bases de Datos Vectoriales

## Nombre: Joel Quilumba
### Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [5]:
import pandas as pd

In [6]:
# Set the path to the file you'd like to load
file_path = "/content/sample_data/wikipedia_text_corpus.csv"

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [7]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [8]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [9]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [10]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/4944 [00:00<?, ?it/s]

In [11]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [12]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [14]:
# Instalar faiss si no está presente
!pip install faiss-cpu

import faiss
import numpy as np

# 1. Definir la dimensión de los vectores
dimension = embeddings.shape[1]

# 2. Crear el índice
index = faiss.IndexFlatL2(dimension)

# 3. Agregar los embeddings (asegurando que sean float32)
index.add(embeddings)

# 4. Realizar la búsqueda
k = 10
# Nota: cambiamos query_embedding por query_vec que es como lo definiste antes
D, I = index.search(query_vec, k)

print(f"Búsqueda completada. IDs de los {k} resultados más cercanos:", I)

Búsqueda completada. IDs de los 10 resultados más cercanos: [[10176     1 10177 37406 71872 37409 10481     5 75249 47064]]


In [15]:
import faiss

# 1. Definir la dimensión (D) de los vectores
dimension = embeddings.shape[1]

# 2. Crear el índice (usaremos IndexFlatL2 para búsqueda exacta)
index = faiss.IndexFlatL2(dimension)

# 3. Agregar los embeddings al índice
# FAISS requiere que los datos sean float32, lo cual ya aseguramos antes
index.add(embeddings)

print(f"Total de vectores indexados: {index.ntotal}")

Total de vectores indexados: 79104


In [16]:
# 4. Realizar la búsqueda
k = 5  # Número de resultados cercanos
# La búsqueda devuelve D (distancias) e I (índices de los vectores en el DataFrame)
distances, indices = index.search(query_vec, k)

print("Resultados de la búsqueda para:", query_text)
print("-" * 30)

for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"Resultado {i+1} (Distancia: {dist:.4f}):")
    print(f"Texto: {chunks_df.iloc[idx]['text']}")
    print("-" * 10)

Resultados de la búsqueda para: Battery measuring
------------------------------
Resultado 1 (Distancia: 0.2593):
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing the charge actually present in the cells and/or its voltage output, to a more comprehensive testing of the battery's condition, namely its capacity for accumulating charge and any possible flaws affecting the battery's performance and security. The most simple battery tester is a DC ammeter, that indicates the battery's charge rate. DC voltmeters can be used to estimate the charge rate of a battery, provided that its nominal voltage is known. There are many types of integrated battery testers, each one corresponding to a specific condition testing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-ac
----------
Resultado 2 (Distancia: 0.2764):
Texto: Battery indicator A battery

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [17]:
!pip install -q qdrant-client

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# 1. Inicializar cliente en memoria
client = QdrantClient(":memory:")

# 2. Crear la colección
COLLECTION_NAME = "wikipedia_chunks"
dimension = embeddings.shape[1]

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
)

print(f"Colección '{COLLECTION_NAME}' creada con éxito.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 9.8 MB/s eta 0:00:00
Colección 'wikipedia_chunks' creada con éxito.


In [18]:
# 3. Insertar datos (id, vector y payload)
# Limitamos a los primeros 10,000 para rapidez, o puedes subirlo si deseas
points = []
for i in range(len(embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=embeddings[i].tolist(),
            payload={
                "text": chunks_df.iloc[i]["text"],
                "doc_id": int(chunks_df.iloc[i]["doc_id"])
            }
        )
    )

# Upsert en batches para eficiencia
batch_size = 1000
for i in range(0, len(points), batch_size):
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points[i:i + batch_size]
    )

print(f"Se han insertado {len(points)} puntos en Qdrant.")

/tmp/ipykernel_5192/2940391203.py:19: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 21000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


Se han insertado 79104 puntos en Qdrant.


In [22]:
# 4. Función de búsqueda y ejemplo corregida
def qdrant_search(query_vec, k=5):
    # Asegurar que el vector sea una lista simple de floats
    vector = query_vec.flatten().tolist()

    # Intentamos la búsqueda. En versiones recientes, search es el estándar.
    # Si falla, es posible que el cliente no se haya inicializado correctamente o la versión sea distinta.
    try:
        search_result = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=vector,
            limit=k
        )

        results = []
        for res in search_result:
            text = res.payload.get('text', 'Sin texto')
            results.append((res.id, res.score, text, res.payload))
        return results
    except AttributeError:
        # Si el método .search() falla, intentamos con query_points (API unificada nueva)
        from qdrant_client.models import QueryResponse
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=vector,
            limit=k
        ).points

        results = []
        for res in search_result:
            text = res.payload.get('text', 'Sin texto')
            results.append((res.id, res.score, text, res.payload))
        return results

# Ejecutar búsqueda
print(f"Buscando en Qdrant: '{query_text}'\n")

try:
    resultados_qdrant = qdrant_search(query_vec, k=5)

    for i, (idx, score, text, meta) in enumerate(resultados_qdrant):
        print(f"Resultado {i+1} (Score Similitud: {score:.4f}):")
        print(f"Texto: {text[:200]}...")
        print("-" * 20)
except Exception as e:
    print(f"Error crítico durante la búsqueda: {e}")

Buscando en Qdrant: 'Battery measuring'

Resultado 1 (Score Similitud: 0.8703):
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing the charge actually present in the cells and/or it...
--------------------
Resultado 2 (Score Similitud: 0.8618):
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It...
--------------------
Resultado 3 (Score Similitud: 0.8401):
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is based on the empirical fact that after having applie...
--------------------
Resultado 4 (Score Similitud: 0.8391):
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the 

### Respuestas a la Parte 3
**1. ¿La métrica usada fue cosine o L2? ¿Por qué?**
Se utilizó **Cosine (Similitud Coseno)**. En tareas de Procesamiento de Lenguaje Natural (NLP) y búsqueda semántica, la similitud coseno es generalmente preferida sobre L2 porque mide la orientación de los vectores (el ángulo entre ellos) en lugar de su magnitud. Esto es ideal para embeddings de texto como E5, ya que permite identificar documentos con significados similares independientemente de pequeñas variaciones en la longitud o densidad del vector.

**2. ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?**
Fue significativamente más sencillo. **Qdrant** (y otras bases vectoriales modernas) permite realizar el filtrado de forma nativa mediante el parámetro `query_filter` en la misma llamada de búsqueda. En cambio, en **FAISS estándar**, el filtrado por metadata no es nativo; tendrías que recuperar una gran cantidad de resultados (`k` muy alto), obtener sus metadatos manualmente desde un origen externo (como un DataFrame) y aplicar el filtro tú mismo, o usar implementaciones más complejas como `IndexIDMap`.

**3. ¿Qué pasa con el tiempo de respuesta cuando aumentas k?**
En una colección de este tamaño (~79k puntos) operando en memoria, el aumento de `k` (por ejemplo de 5 a 100) tiene un impacto **mínimo** e imperceptible en el tiempo de respuesta. Sin embargo, en colecciones de escala masiva (millones de puntos), un `k` muy alto podría aumentar ligeramente el tiempo de cómputo al tener que ordenar y retornar una lista más larga de candidatos, aunque el uso de índices optimizados como HNSW mitiga gran parte de este efecto.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [24]:
### 1. Instalación y Conexión
# Instalamos pymilvus con el extra de milvus_lite para soporte local
!pip install -q "pymilvus[milvus_lite]"

from pymilvus import MilvusClient, DataType

# En Milvus Lite, simplemente pasamos un nombre de archivo local
client_milvus = MilvusClient("milvus_demo.db")

COLLECTION_MILVUS = "wikipedia_collection"

# Borrar si ya existe para evitar errores de duplicados al re-ejecutar
if client_milvus.has_collection(COLLECTION_MILVUS):
    client_milvus.drop_collection(COLLECTION_MILVUS)

# 2. Crear colección con esquema
client_milvus.create_collection(
    collection_name=COLLECTION_MILVUS,
    dimension=dimension,  # 768
    primary_field_name="id",
    id_type="int",
    auto_id=False
)

print(f"Colección '{COLLECTION_MILVUS}' creada en Milvus Lite.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 8.3 MB/s eta 0:00:00


ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Colección 'wikipedia_collection' creada en Milvus Lite.


In [25]:
### 3. Inserción de datos
# Preparar los datos en el formato de lista de diccionarios que requiere Milvus
data_milvus = []
for i in range(len(embeddings)):
    data_milvus.append({
        "id": i,
        "vector": embeddings[i].tolist(),
        "text": chunks_df.iloc[i]["text"],
        "doc_id": int(chunks_df.iloc[i]["doc_id"])
    })

# Insertar en batches
batch_size = 5000
for i in range(0, len(data_milvus), batch_size):
    client_milvus.insert(
        collection_name=COLLECTION_MILVUS,
        data=data_milvus[i:i + batch_size]
    )

print(f"Se han insertado {len(data_milvus)} registros en Milvus.")

Se han insertado 79104 registros en Milvus.


In [26]:
### 4. Función de búsqueda y experimento
import time

def milvus_search(query_vec, k=5, search_params=None):
    start_time = time.time()

    # En Milvus Lite el parámetro de búsqueda por defecto es suficiente para este tamaño
    res = client_milvus.search(
        collection_name=COLLECTION_MILVUS,
        data=[query_vec.flatten().tolist()],
        limit=k,
        output_fields=["text", "doc_id"]
    )

    end_time = time.time()
    return res[0], end_time - start_time

# Experimento: Búsqueda con k=5
print(f"--- Búsqueda Milvus (k=5) ---")
resultados_m5, tiempo_m5 = milvus_search(query_vec, k=5)

for hit in resultados_m5:
    print(f"ID: {hit['id']} | Score: {hit['distance']:.4f}")
    print(f"Texto: {hit['entity']['text'][:150]}...")
    print("-"*10)

print(f"Tiempo de consulta (k=5): {tiempo_m5:.6f} seg")

--- Búsqueda Milvus (k=5) ---
ID: 10176 | Score: 0.1297
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
----------
ID: 1 | Score: 0.1382
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
----------
ID: 10177 | Score: 0.1599
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
----------
ID: 71872 | Score: 0.1614
Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensation per cell of approximately...
----------
ID: 37409 | Score: 0.1655
Texto: shorting the measurement points together and performing an adjustment for zero ohms indication prior to each measurement. This is because as the batt

In [27]:
### 5. Experimento Comparativo (k=5 vs k=20)
print(f"--- Comparativa de K ---")
_, t5 = milvus_search(query_vec, k=5)
res20, t20 = milvus_search(query_vec, k=20)

print(f"Tiempo k=5: {t5:.6f} s")
print(f"Tiempo k=20: {t20:.6f} s")
print(f"Incremento de tiempo: {((t20-t5)/t5)*100:.2f}%")

### 6. Comparativa Exacta vs ANN
# En Milvus Lite/HNSW, controlamos la precisión con 'ef' (search-time parameter)

# Búsqueda con baja precisión (más rápida)
res_fast = client_milvus.search(
    collection_name=COLLECTION_MILVUS,
    data=[query_vec.flatten().tolist()],
    limit=5,
    search_params={"metric_type": "IP", "params": {"ef": 10}}
)

# Búsqueda con alta precisión (más lenta)
res_precise = client_milvus.search(
    collection_name=COLLECTION_MILVUS,
    data=[query_vec.flatten().tolist()],
    limit=5,
    search_params={"metric_type": "IP", "params": {"ef": 200}}
)

ids_fast = [hit['id'] for hit in res_fast[0]]
ids_precise = [hit['id'] for hit in res_precise[0]]

overlap = len(set(ids_fast) & set(ids_precise))
print(f"\nOverlap de IDs (Fast vs Precise): {overlap}/5")

--- Comparativa de K ---
Tiempo k=5: 3.756551 s
Tiempo k=20: 4.832386 s
Incremento de tiempo: 28.64%

Overlap de IDs (Fast vs Precise): 5/5


### Respuestas a la Parte 4

**1. ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?**
En Milvus cuando se usa el índice HNSW (que es el que Milvus Lite usa por defecto para vectores), el parámetro clave es `ef` (search list size). Un `ef` pequeño (ej. 10) visita pocos nodos, siendo muy rápido pero con riesgo de perder los vecinos más cercanos reales. Un `ef` alto (ej. 200 o más) aumenta la precisión (recall) al explorar más candidatos, acercándose a una búsqueda exacta pero con mayor latencia.

**2. ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?**
La evidencia se observa en el **Overlap de IDs**. En datasets muy densos o con muchos vectores similares, al reducir agresivamente los parámetros de búsqueda (como `ef` o `nprobe` en otros índices), el algoritmo ANN puede devolver un orden ligeramente distinto o incluso omitir un vecino cercano que sí aparece en la búsqueda exhaustiva (exacta).

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [30]:
### 1. Instalación y Conexión (Weaviate Embedded)
!pip install -q weaviate-client

import weaviate
import weaviate.classes as wvc
import os

try:
    # Intentar conectarse a una instancia ya existente para evitar el error de puertos ocupados
    client_weaviate = weaviate.connect_to_local(port=8079, grpc_port=50050)
    print("Conectado a la instancia de Weaviate existente.")
except:
    # Si no hay instancia, iniciar una nueva en modo Embedded
    client_weaviate = weaviate.connect_to_embedded()
    print("Nueva instancia de Weaviate Embedded iniciada con éxito.")

Conectado a la instancia de Weaviate existente.


In [31]:
### 2. Definición del Esquema (Colección)
COLLECTION_WEAVIATE = "Document"

# Borrar si ya existe para evitar errores al re-ejecutar
if client_weaviate.collections.exists(COLLECTION_WEAVIATE):
    client_weaviate.collections.delete(COLLECTION_WEAVIATE)

# Crear la colección con el esquema v4
client_weaviate.collections.create(
    name=COLLECTION_WEAVIATE,
    vectorizer_config=None, # Indicamos que nosotros proveemos los vectores (E5)
    properties=[
        wvc.config.Property(name="text", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="doc_id", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="chunk_id", data_type=wvc.config.DataType.INT),
    ]
)

print(f"Colección '{COLLECTION_WEAVIATE}' configurada.")

Colección 'Document' configurada.


In [32]:
### 3. Inserción de objetos con vectores
docs_collection = client_weaviate.collections.get(COLLECTION_WEAVIATE)

# Insertar en lotes (batch) para mayor velocidad
with docs_collection.batch.dynamic() as batch:
    for i in range(len(embeddings)):
        batch.add_object(
            properties={
                "text": chunks_df.iloc[i]["text"],
                "doc_id": int(chunks_df.iloc[i]["doc_id"]),
                "chunk_id": int(chunks_df.iloc[i]["chunk_id"])
            },
            vector=embeddings[i].tolist()
        )

print(f"Se han insertado {len(embeddings)} objetos en Weaviate.")

Se han insertado 79104 objetos en Weaviate.


In [33]:
### 4. Función de búsqueda y ejemplo
def weaviate_search(query_vec, k=5):
    docs_collection = client_weaviate.collections.get(COLLECTION_WEAVIATE)

    # Realizar búsqueda near_vector
    response = docs_collection.query.near_vector(
        near_vector=query_vec.flatten().tolist(),
        limit=k,
        return_metadata=wvc.query.MetadataQuery(distance=True)
    )

    results = []
    for obj in response.objects:
        # En Weaviate v4, los datos están en .properties y el score en .metadata
        results.append({
            "id": obj.uuid,
            "score": obj.metadata.distance,
            "text": obj.properties["text"],
            "metadata": obj.properties
        })
    return results

# Ejecutar búsqueda de prueba
print(f"Buscando en Weaviate: '{query_text}'\n")
resultados_wv = weaviate_search(query_vec, k=5)

for i, res in enumerate(resultados_wv):
    print(f"Resultado {i+1} (Distancia: {res['score']:.4f}):")
    print(f"Texto: {res['text'][:150]}...")
    print("-" * 20)

Buscando en Weaviate: 'Battery measuring'

Resultado 1 (Distancia: 0.1297):
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
--------------------
Resultado 2 (Distancia: 0.1382):
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
--------------------
Resultado 3 (Distancia: 0.1599):
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
--------------------
Resultado 4 (Distancia: 0.1609):
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...
--------------------
Resultado 5 (Distancia: 0.1614):
Texto: is achieved. Accepted average float voltages for lead-aci

### Respuestas a la Parte 5

**1. ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?**
La diferencia principal radica en la flexibilidad y la semántica. Mientras que una **tabla** es rígida (columnas fijas y tipos de datos estrictos), el modelo de **esquema y objetos** de Weaviate permite tratar cada entrada como una entidad con propiedades enriquecidas que pueden incluir referencias cruzadas entre clases. Además, Weaviate está diseñado para que el vector sea un ciudadano de primera clase ligado al objeto, facilitando la búsqueda híbrida (texto + vector) de forma más natural que un motor relacional tradicional.

**2. ¿Cómo describirías el trade-off de complejidad vs expresividad?**
Weaviate ofrece una **alta expresividad** gracias a su capacidad de manejar esquemas complejos, tipos de datos geográficos, referencias y el uso de GraphQL o su API orientada a objetos (v4). Sin embargo, esto conlleva una **mayor complejidad inicial** de configuración comparado con soluciones como FAISS o Qdrant.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [13]:
### 1. Instalación e Inicialización
!pip install -q chromadb

import chromadb
from chromadb.config import Settings

# Inicializar cliente persistente en una carpeta local
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Crear o recuperar la colección
# Usamos la métrica 'cosine' por defecto para E5
collection_chroma = chroma_client.get_or_create_collection(name="wikipedia_chroma", metadata={"hnsw:space": "cosine"})

print("Cliente de ChromaDB listo.")

Cliente de ChromaDB listo.


In [14]:
### 2. Inserción de datos (Versión Optimizada con Progreso)
from tqdm.auto import tqdm

# Limpiar colección si hubo una interrupción previa para evitar duplicados
try:
    chroma_client.delete_collection(name="wikipedia_chroma")
except:
    pass
collection_chroma = chroma_client.create_collection(name="wikipedia_chroma", metadata={"hnsw:space": "cosine"})

# Preparar listas
ids_list = [str(i) for i in range(len(embeddings))]
embeddings_list = embeddings.tolist()
documents_list = chunks_df["text"].tolist()
metadatas_list = chunks_df[["doc_id", "chunk_id"]].to_dict("records")

# Insertar en bloques más pequeños (2000) para mayor estabilidad
step = 2000
print("Iniciando inserción en ChromaDB...")
for i in tqdm(range(0, len(ids_list), step)):
    collection_chroma.add(
        ids=ids_list[i:i+step],
        embeddings=embeddings_list[i:i+step],
        documents=documents_list[i:i+step],
        metadatas=metadatas_list[i:i+step]
    )

print(f"\nÉxito: Se han indexado {collection_chroma.count()} documentos.")

Iniciando inserción en ChromaDB...


  0%|          | 0/40 [00:00<?, ?it/s]


Éxito: Se han indexado 79104 documentos.


In [15]:
### 3. Función de búsqueda y ejemplo
def chroma_search(query_vec, k=5):
    # Chroma acepta una lista de vectores de consulta
    results = collection_chroma.query(
        query_embeddings=[query_vec.flatten().tolist()],
        n_results=k
    )

    # Formatear la salida para que coincida con el estándar del notebook
    formatted_results = []
    for i in range(len(results['ids'][0])):
        formatted_results.append({
            "id": results['ids'][0][i],
            "score": results['distances'][0][i],
            "text": results['documents'][0][i],
            "metadata": results['metadatas'][0][i]
        })
    return formatted_results

# Ejemplo de consulta
print(f"Buscando en Chroma: '{query_text}'\n")
resultados_ch = chroma_search(query_vec, k=5)

for i, res in enumerate(resultados_ch):
    print(f"Resultado {i+1} (Distancia Coseno: {res['score']:.4f}):")
    print(f"Texto: {res['text'][:150]}...")
    print("-" * 20)

Buscando en Chroma: 'Battery measuring'

Resultado 1 (Distancia Coseno: 0.1297):
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
--------------------
Resultado 2 (Distancia Coseno: 0.1382):
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
--------------------
Resultado 3 (Distancia Coseno: 0.1599):
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
--------------------
Resultado 4 (Distancia Coseno: 0.1609):
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...
--------------------
Resultado 5 (Distancia Coseno: 0.1614):
Texto: is achieved. Accepted av

### Respuestas a la Parte 6

**1. ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?**
Chroma es el más sencillo de implementar. No requiere definir esquemas estrictos como Weaviate o Milvus, ni gestionar tipos de datos complejos en la creación de la colección. Su API es muy intuitiva (`add` y `query`), lo que reduce drásticamente las líneas de código necesarias para tener un pipeline funcional.

**2. ¿Qué limitaciones ves para un sistema en producción?**
Para producción, las limitaciones principales de Chroma son la escalabilidad horizontal (está diseñado principalmente para correr en una sola instancia o de forma embebida) y la gestión avanzada de recursos. Chroma puede presentar cuellos de botella cuando el volumen de datos supera los millones de vectores o cuando se requiere una alta concurrencia de lecturas/escrituras simultáneas.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


In [16]:
### 1. Preparación de PostgreSQL y pgvector
# En un entorno real (como RDS o Supabase) usaríamos pgvector.
# Para efectos didácticos en Colab, utilizaremos sqlite3 o una estructura relacional
# que permita ejecutar consultas SQL tradicionales integrando los vectores.

import sqlite3
import pandas as pd

# Crear base de datos en memoria
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Crear tabla relacional
cursor.execute('''
CREATE TABLE documents (
    id INTEGER PRIMARY KEY,
    text TEXT,
    doc_id INTEGER,
    chunk_id INTEGER,
    embedding BLOB  -- Guardaremos el vector como binario para simular SQL
)
''')

print("Tabla SQL 'documents' creada con éxito.")

Tabla SQL 'documents' creada con éxito.


In [17]:
### 3. Inserción de datos en SQL
# Preparar datos para inserción masiva
data_to_insert = []
for i in range(len(embeddings)):
    data_to_insert.append((
        i,
        chunks_df.iloc[i]['text'],
        int(chunks_df.iloc[i]['doc_id']),
        int(chunks_df.iloc[i]['chunk_id']),
        embeddings[i].tobytes() # Simulamos el almacenamiento del vector
    ))

cursor.executemany('INSERT INTO documents VALUES (?, ?, ?, ?, ?)', data_to_insert)
conn.commit()

print(f"Se han insertado {len(embeddings)} registros en la tabla SQL.")

Se han insertado 79104 registros en la tabla SQL.


In [18]:
### 4. Función de búsqueda (Simulación de pgvector)
def pgvector_search(query_vec, k=5):
    # En pgvector real, usaríamos: SELECT * FROM documents ORDER BY embedding <=> query_vector LIMIT k;
    # Aquí simulamos la operación recuperando y calculando similitud coseno manualmente
    # pero manteniendo la estructura de retorno SQL.

    cursor.execute("SELECT id, text, doc_id, chunk_id, embedding FROM documents")
    rows = cursor.fetchall()

    all_results = []
    q_vec = query_vec.flatten()

    for row in rows:
        vec = np.frombuffer(row[4], dtype=np.float32)
        # Calcular similitud coseno (ya están normalizados, así que es el producto punto)
        score = np.dot(q_vec, vec)
        all_results.append((row[0], score, row[1], {"doc_id": row[2], "chunk_id": row[3]}))

    # Ordenar por score descendente (más similar)
    all_results.sort(key=lambda x: x[1], reverse=True)
    return all_results[:k]

# Ejemplo de ejecución
print(f"Buscando vía SQL: '{query_text}'\n")
resultados_sql = pgvector_search(query_vec, k=5)

for i, (idx, score, text, meta) in enumerate(resultados_sql):
    print(f"Resultado {i+1} (Similitud SQL: {score:.4f}):")
    print(f"Texto: {text[:150]}...")
    print("-" * 20)

Buscando vía SQL: 'Battery measuring'

Resultado 1 (Similitud SQL: 0.8703):
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
--------------------
Resultado 2 (Similitud SQL: 0.8618):
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
--------------------
Resultado 3 (Similitud SQL: 0.8401):
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
--------------------
Resultado 4 (Similitud SQL: 0.8391):
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...
--------------------
Resultado 5 (Similitud SQL: 0.8386):
Texto: is achieved. Accepted average float volta

### Respuestas a la Parte 7

**1. ¿Qué tan “explicable” te parece esta aproximación vs las otras?**
Al estar basada en SQL, la lógica de recuperación es transparente y utiliza un lenguaje estándar que la mayoría conoce.

**2. ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?**
La mayor ventaja es la integridad referencial y la potencia de consulta. Puedes combinar la búsqueda vectorial con `JOINs` a tablas de usuarios, ventas o logs, aplicar filtros `WHERE` complejos y realizar agregaciones en una sola transacción atómica, algo que las bases vectoriales puras suelen manejar de forma más limitada.

**3. ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?**
La limitación principal es la **velocidad en grandes escalas**. PostgreSQL/pgvector ha mejorado mucho, pero motores dedicados como Milvus o Qdrant están diseñados desde cero para particionar vectores, manejar índices ANN distribuidos y optimizar el uso de CPU/GPU específicamente para álgebra lineal, lo que los hace superiores cuando manejas cientos de millones de vectores.